# Import Data and Cleaning

In [2]:
# Import libraries
import numpy as np
import pandas as pd
import time

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report, recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb

In [3]:
df = pd.read_csv("data/online_shopping_cleaned.csv")
df

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
0,Returning_Visitor,0.0,0.200000,0.000000,1.0,1,0.200000,0.000000,0
1,Returning_Visitor,0.0,0.100000,0.000000,2.0,1,0.000000,64.000000,0
2,Returning_Visitor,NaN,0.200000,0.000000,3.0,9,0.200000,0.000000,0
3,Returning_Visitor,0.0,0.140000,0.000000,4.0,2,0.050000,2.666667,0
4,Returning_Visitor,0.0,NaN,NaN,4.0,1,0.020000,627.500000,0
...,...,...,...,...,...,...,...,...,...
12325,Returning_Visitor,0.0,0.029031,12.241717,NaN,1,0.007143,1783.791667,0
12326,Returning_Visitor,0.0,0.021333,NaN,8.0,1,0.000000,465.750000,0
12327,Returning_Visitor,0.0,0.086667,0.000000,13.0,1,0.083333,184.250000,0
12328,Returning_Visitor,0.0,0.021053,0.000000,11.0,3,0.000000,346.000000,0


In [4]:
df.drop_duplicates(inplace=True)
df

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
0,Returning_Visitor,0.0,0.200000,0.000000,1.0,1,0.200000,0.000000,0
1,Returning_Visitor,0.0,0.100000,0.000000,2.0,1,0.000000,64.000000,0
2,Returning_Visitor,NaN,0.200000,0.000000,3.0,9,0.200000,0.000000,0
3,Returning_Visitor,0.0,0.140000,0.000000,4.0,2,0.050000,2.666667,0
4,Returning_Visitor,0.0,NaN,NaN,4.0,1,0.020000,627.500000,0
...,...,...,...,...,...,...,...,...,...
12325,Returning_Visitor,0.0,0.029031,12.241717,NaN,1,0.007143,1783.791667,0
12326,Returning_Visitor,0.0,0.021333,NaN,8.0,1,0.000000,465.750000,0
12327,Returning_Visitor,0.0,0.086667,0.000000,13.0,1,0.083333,184.250000,0
12328,Returning_Visitor,0.0,0.021053,0.000000,11.0,3,0.000000,346.000000,0


In [146]:
df.info()

<class 'pandas.DataFrame'>
Index: 11845 entries, 0 to 12329
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerType         11845 non-null  str    
 1   SpecialDayProximity  11845 non-null  float64
 2   ExitRate             11845 non-null  float64
 3   PageValue            11845 non-null  float64
 4   TrafficSource        11845 non-null  float64
 5   GeographicRegion     11845 non-null  int64  
 6   BounceRate           11845 non-null  float64
 7   ProductPageTime      11845 non-null  float64
 8   PurchaseCompleted    11845 non-null  int64  
dtypes: float64(6), int64(2), str(1)
memory usage: 925.4 KB


In [147]:
df.describe(include="all")

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
count,11845,11845.000000,11845.000000,11845.000000,11845.000000,11845.00000,11845.000000,11845.000000,11845.000000
unique,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,Returning_Visitor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,9562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.059316,0.036034,5.840929,3.967497,3.15745,0.015226,1216.938805,0.161081
std,NaN,0.195414,0.036575,18.589012,3.925498,2.40144,0.033971,1907.759372,0.367621
min,NaN,0.000000,0.000000,0.000000,1.000000,1.00000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.014286,0.000000,2.000000,1.00000,0.000000,237.750000,0.000000
50%,NaN,0.000000,0.025286,0.000000,2.000000,3.00000,0.002406,664.800000,0.000000
75%,NaN,0.000000,0.042320,0.000000,4.000000,4.00000,0.014516,1446.733333,0.000000


In [5]:
df.dropna(inplace=True)
df

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
0,Returning_Visitor,0.0,0.200000,0.0,1.0,1,0.200000,0.000000,0
1,Returning_Visitor,0.0,0.100000,0.0,2.0,1,0.000000,64.000000,0
3,Returning_Visitor,0.0,0.140000,0.0,4.0,2,0.050000,2.666667,0
5,Returning_Visitor,0.0,0.024561,0.0,3.0,1,0.015789,154.216667,0
6,Returning_Visitor,0.4,0.200000,0.0,3.0,3,0.200000,0.000000,0
...,...,...,...,...,...,...,...,...,...
12323,Returning_Visitor,0.0,0.013953,0.0,10.0,1,0.000000,1157.976190,0
12324,Returning_Visitor,0.0,0.037647,0.0,1.0,1,0.000000,503.000000,0
12327,Returning_Visitor,0.0,0.086667,0.0,13.0,1,0.083333,184.250000,0
12328,Returning_Visitor,0.0,0.021053,0.0,11.0,3,0.000000,346.000000,0


In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 9177 entries, 0 to 12329
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerType         9177 non-null   str    
 1   SpecialDayProximity  9177 non-null   float64
 2   ExitRate             9177 non-null   float64
 3   PageValue            9177 non-null   float64
 4   TrafficSource        9177 non-null   float64
 5   GeographicRegion     9177 non-null   int64  
 6   BounceRate           9177 non-null   float64
 7   ProductPageTime      9177 non-null   float64
 8   PurchaseCompleted    9177 non-null   int64  
dtypes: float64(6), int64(2), str(1)
memory usage: 717.0 KB


In [ ]:
df.describe(include="all")

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
count,9177,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000
unique,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,Returning_Visitor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,7401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.062243,0.036462,6.220592,4.050561,3.152991,0.014793,1244.123949,0.162144
std,NaN,0.200013,0.037447,19.291962,3.979877,2.398216,0.033065,1969.727163,0.368603
min,NaN,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.013441,0.000000,2.000000,1.000000,0.000000,221.780000,0.000000
50%,NaN,0.000000,0.024815,0.000000,2.000000,3.000000,0.002174,654.066667,0.000000
75%,NaN,0.000000,0.044351,0.000000,4.000000,4.000000,0.014286,1527.583333,0.000000


# Feature Engineering

In [6]:
# New column: has_page_value
df["has_page_value"] = df["PageValue"] > 0
df["has_page_value"] = df["has_page_value"].astype(int)
df["has_page_value"].value_counts()

has_page_value
0    7050
1    2127
Name: count, dtype: int64

In [7]:
# Log transform ProductPageTime and PageValue
log_transformer = FunctionTransformer(np.log1p, validate=True)
df["PageValue_log"] = log_transformer.transform(df[["PageValue"]])
df["ProductPageTime_log"] = log_transformer.transform(df[["ProductPageTime"]])
df.describe(include='all')

c:\Users\kaiho\Downloads\Python\aiap23-low-kai-hon-213I\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but FunctionTransformer was fitted without feature names
  warnings.warn(
c:\Users\kaiho\Downloads\Python\aiap23-low-kai-hon-213I\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but FunctionTransformer was fitted without feature names
  warnings.warn(


,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted,has_page_value,PageValue_log,ProductPageTime_log
count,9177,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000
unique,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,Returning_Visitor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,7401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.062243,0.036462,6.220592,4.050561,3.152991,0.014793,1244.123949,0.162144,0.231775,0.654797,6.208250
std,NaN,0.200013,0.037447,19.291962,3.979877,2.398216,0.033065,1969.727163,0.368603,0.421989,1.291921,1.685346
min,NaN,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.013441,0.000000,2.000000,1.000000,0.000000,221.780000,0.000000,0.000000,0.000000,5.406185
50%,NaN,0.000000,0.024815,0.000000,2.000000,3.000000,0.002174,654.066667,0.000000,0.000000,0.000000,6.484737
75%,NaN,0.000000,0.044351,0.000000,4.000000,4.000000,0.014286,1527.583333,0.000000,0.000000,0.000000,7.332097


# One-hot encode

In [8]:
# For linear models to avoid collinearity
df_drop_first = pd.get_dummies(df, columns=["CustomerType", "TrafficSource", "GeographicRegion"], drop_first=True, dtype='int')
df_drop_first

,SpecialDayProximity,ExitRate,PageValue,BounceRate,ProductPageTime,PurchaseCompleted,has_page_value,PageValue_log,ProductPageTime_log,CustomerType_Other,...,TrafficSource_19.0,TrafficSource_20.0,GeographicRegion_2,GeographicRegion_3,GeographicRegion_4,GeographicRegion_5,GeographicRegion_6,GeographicRegion_7,GeographicRegion_8,GeographicRegion_9
0,0.0,0.200000,0.0,0.200000,0.000000,0,0,0.0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
1,0.0,0.100000,0.0,0.000000,64.000000,0,0,0.0,4.174387,0,...,0,0,0,0,0,0,0,0,0,0
3,0.0,0.140000,0.0,0.050000,2.666667,0,0,0.0,1.299283,0,...,0,0,1,0,0,0,0,0,0,0
5,0.0,0.024561,0.0,0.015789,154.216667,0,0,0.0,5.044822,0,...,0,0,0,0,0,0,0,0,0,0
6,0.4,0.200000,0.0,0.200000,0.000000,0,0,0.0,0.000000,0,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12323,0.0,0.013953,0.0,0.000000,1157.976190,0,0,0.0,7.055292,0,...,0,0,0,0,0,0,0,0,0,0
12324,0.0,0.037647,0.0,0.000000,503.000000,0,0,0.0,6.222576,0,...,0,0,0,0,0,0,0,0,0,0
12327,0.0,0.086667,0.0,0.083333,184.250000,0,0,0.0,5.221706,0,...,0,0,0,0,0,0,0,0,0,0
12328,0.0,0.021053,0.0,0.000000,346.000000,0,0,0.0,5.849325,0,...,0,0,0,1,0,0,0,0,0,0


# Set Predictors and Target

In [9]:
X = df_drop_first.drop(columns='PurchaseCompleted')
# X = df.drop(columns='PurchaseCompleted')  # For lightgbm (categorical) - no one-hot encoding 
X

,SpecialDayProximity,ExitRate,PageValue,BounceRate,ProductPageTime,has_page_value,PageValue_log,ProductPageTime_log,CustomerType_Other,CustomerType_Returning_Visitor,...,TrafficSource_19.0,TrafficSource_20.0,GeographicRegion_2,GeographicRegion_3,GeographicRegion_4,GeographicRegion_5,GeographicRegion_6,GeographicRegion_7,GeographicRegion_8,GeographicRegion_9
0,0.0,0.200000,0.0,0.200000,0.000000,0,0.0,0.000000,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0.0,0.100000,0.0,0.000000,64.000000,0,0.0,4.174387,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0.0,0.140000,0.0,0.050000,2.666667,0,0.0,1.299283,0,1,...,0,0,1,0,0,0,0,0,0,0
5,0.0,0.024561,0.0,0.015789,154.216667,0,0.0,5.044822,0,1,...,0,0,0,0,0,0,0,0,0,0
6,0.4,0.200000,0.0,0.200000,0.000000,0,0.0,0.000000,0,1,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12323,0.0,0.013953,0.0,0.000000,1157.976190,0,0.0,7.055292,0,1,...,0,0,0,0,0,0,0,0,0,0
12324,0.0,0.037647,0.0,0.000000,503.000000,0,0.0,6.222576,0,1,...,0,0,0,0,0,0,0,0,0,0
12327,0.0,0.086667,0.0,0.083333,184.250000,0,0.0,5.221706,0,1,...,0,0,0,0,0,0,0,0,0,0
12328,0.0,0.021053,0.0,0.000000,346.000000,0,0.0,5.849325,0,1,...,0,0,0,1,0,0,0,0,0,0


In [10]:
y = df.PurchaseCompleted
y

0        0
1        0
3        0
5        0
6        0
        ..
12323    0
12324    0
12327    0
12328    0
12329    0
Name: PurchaseCompleted, Length: 9177, dtype: int64

# Split Data

**Use stratify to keep proportions of y the same.**

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1, stratify = y)
print(X_train.shape, "\n", X_train[:5], "\n")
print(X_test.shape, "\n", X_test[:5], "\n")
print(y_train.shape, "\n", y_train[:5], "\n")
print(y_test.shape, "\n", y_test[:5])

(7341, 38) 
       SpecialDayProximity  ExitRate  PageValue  BounceRate  ProductPageTime  \
5429                  0.0  0.116000   0.000000    0.080000       334.000000   
6015                  0.0  0.009375   0.000000    0.004167      3025.333333   
9563                  0.0  0.019022  10.519018    0.002020      2434.144872   
5241                  0.6  0.028571   0.000000    0.000000       486.000000   
4205                  0.0  0.058333   0.000000    0.011111       110.000000   

      has_page_value  PageValue_log  ProductPageTime_log  CustomerType_Other  \
5429               0       0.000000             5.814131                   0   
6015               0       0.000000             8.015107                   0   
9563               1       2.443999             7.797762                   0   
5241               0       0.000000             6.188264                   0   
4205               0       0.000000             4.709530                   0   

      CustomerType_Returning_Vi

In [272]:
y_train.value_counts(normalize=True)

PurchaseCompleted
0    0.837897
1    0.162103
Name: proportion, dtype: float64

In [273]:
y_test.value_counts(normalize=True)

PurchaseCompleted
0    0.837691
1    0.162309
Name: proportion, dtype: float64

# Normalise Data

In [ ]:
# Apply StandardScaler to continuous features: ExitRate, PageValue, BounceRate, ProductPageTime (leave SpecialDayProximity as is - it's already bounded between 0 and 1)

# scale_features = ['ExitRate', 'PageValue', 'BounceRate', 'ProductPageTime']

# With feature engineered features
scale_features = ['ExitRate', 'PageValue', 'BounceRate', 'ProductPageTime', 'PageValue_log', 'ProductPageTime_log']

passthrough_features = X.drop(columns=scale_features).columns.tolist()

scaler = ColumnTransformer(
    transformers=[
        ('scale', StandardScaler(), scale_features)
    ],
    remainder='passthrough'
)

In [303]:
# fit transform X_train
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=scale_features+passthrough_features)

binary_features = passthrough_features.copy()
binary_features.remove('SpecialDayProximity')  # Continuous feature

# Convert binary features back to int type
for col in binary_features:
    X_train_scaled[col] = X_train_scaled[col].astype('int')

X_train_scaled

,ExitRate,BounceRate,PageValue_log,ProductPageTime_log,SpecialDayProximity,has_page_value,CustomerType_Other,CustomerType_Returning_Visitor,CustomerType_Unknown,TrafficSource_2.0,...,TrafficSource_19.0,TrafficSource_20.0,GeographicRegion_2,GeographicRegion_3,GeographicRegion_4,GeographicRegion_5,GeographicRegion_6,GeographicRegion_7,GeographicRegion_8,GeographicRegion_9
0,-0.632774,-0.276196,-0.492037,0.166850,0.0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
1,-0.783959,-0.449549,-0.492037,0.205481,0.0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
2,-0.376715,-0.449549,-0.492037,-0.029647,0.0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3.135770,2.514797,-0.492037,-0.221599,0.0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0.130109,-0.023637,-0.492037,0.718678,0.8,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9471,-0.798002,-0.449549,1.688338,0.570137,0.0,1,0,1,0,1,...,0,0,0,0,0,0,0,0,1,0
9472,-0.859726,-0.449549,2.754792,0.816972,0.0,1,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
9473,2.433273,2.020739,-0.492037,-0.806644,0.0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
9474,-0.118676,-0.296221,-0.492037,0.847005,0.0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0


In [304]:
# Only transform X_test
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=scale_features+passthrough_features)

for col in binary_features:
    X_test_scaled[col] = X_test_scaled[col].astype('int')

X_test_scaled

,ExitRate,BounceRate,PageValue_log,ProductPageTime_log,SpecialDayProximity,has_page_value,CustomerType_Other,CustomerType_Returning_Visitor,CustomerType_Unknown,TrafficSource_2.0,...,TrafficSource_19.0,TrafficSource_20.0,GeographicRegion_2,GeographicRegion_3,GeographicRegion_4,GeographicRegion_5,GeographicRegion_6,GeographicRegion_7,GeographicRegion_8,GeographicRegion_9
0,-0.487781,-0.449549,-0.492037,-0.316114,0.0,0,0,1,0,1,...,0,0,0,0,1,0,0,0,0,0
1,-0.608214,-0.449549,-0.492037,0.699636,0.0,0,0,1,0,1,...,0,0,0,0,0,1,0,0,0,0
2,-0.651521,-0.364854,1.256458,0.358238,0.0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,-0.877626,-0.449549,-0.492037,-0.181981,0.4,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0.845019,-0.449549,-0.492037,-1.818428,0.0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2364,-0.584409,-0.251926,-0.492037,0.555065,0.0,0,0,1,0,1,...,0,0,0,1,0,0,0,0,0,0
2365,-0.682148,-0.449549,1.875752,0.144447,0.0,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2366,-0.291872,-0.083581,-0.492037,0.861862,0.6,0,0,1,0,1,...,0,0,0,1,0,0,0,0,0,0
2367,-0.513205,-0.378970,-0.492037,1.041150,0.0,0,0,1,0,1,...,0,0,0,1,0,0,0,0,0,0


# Modelling

In [305]:
# helper function to evaluate model performance
def evaluate(model):
    y_pred_class = model.predict(X_test_scaled)
    y_pred_prob = model.predict_proba(X_test_scaled)
    accuracy = model.score(X_test_scaled, y_test)
    roc_auc = roc_auc_score(y_test, y_pred_prob[:,1])
    f1 = f1_score(y_test, y_pred_class)
    
    print("Train Accuracy: ", model.score(X_train_scaled, y_train))
    print("Test Accuracy: ", accuracy)
    print("confusion_matrix:\n", confusion_matrix(y_test, y_pred_class))
    print("roc_auc_score: ", roc_auc)
    print("classification_report:\n", classification_report(y_test, y_pred_class))
    
    return accuracy, recall_score(y_test, y_pred_class), precision_score(y_test, y_pred_class), roc_auc, f1

In [ ]:
# Non-scaled
def evaluate(model):
    y_pred_class = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)
    accuracy = model.score(X_test, y_test)
    roc_auc = roc_auc_score(y_test, y_pred_prob[:,1])
    f1 = f1_score(y_test, y_pred_class)
    
    print("Train Accuracy: ", model.score(X_train, y_train))
    print("Test Accuracy: ", accuracy)
    print("confusion_matrix:\n", confusion_matrix(y_test, y_pred_class))
    print("roc_auc_score: ", roc_auc)
    print("classification_report:\n", classification_report(y_test, y_pred_class))
    
    return accuracy, recall_score(y_test, y_pred_class), precision_score(y_test, y_pred_class), roc_auc, f1

In [306]:
results_df = pd.DataFrame(columns = ['accuracy', 'recall', 'precision', 'f1_score', 'roc_auc', 'time_taken (s)'])

# helper function to save results
def save_results(model_name, accuracy, recall, precision, f1, roc_auc, time_taken):
    results_df.loc[model_name] = {'accuracy': accuracy, 'recall': recall, 'precision': precision, 'f1_score': f1, 'roc_auc': roc_auc, 'time_taken (s)': time_taken}

## Logistic Regression

In [307]:
start_time = time.perf_counter()
lr = LogisticRegression(max_iter=1000, random_state=1).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(lr)
save_results('Logistic Regression', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8855002110595188
Test Accuracy:  0.888560574081891
confusion_matrix:
 [[1887  100]
 [ 164  218]]
roc_auc_score:  0.8830381774729458
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.95      0.93      1987
           1       0.69      0.57      0.62       382

    accuracy                           0.89      2369
   macro avg       0.80      0.76      0.78      2369
weighted avg       0.88      0.89      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918


In [308]:
start_time = time.perf_counter()
lr = LogisticRegression(max_iter=1000, random_state=1, class_weight='balanced').fit(X_train_scaled, y_train)  # class_weight='balanced'
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(lr)
save_results('Logistic Regression (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.86639932460954
Test Accuracy:  0.8699873364288729
confusion_matrix:
 [[1769  218]
 [  90  292]]
roc_auc_score:  0.8831356698118926
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.89      0.92      1987
           1       0.57      0.76      0.65       382

    accuracy                           0.87      2369
   macro avg       0.76      0.83      0.79      2369
weighted avg       0.89      0.87      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105


## SVM

In [309]:
start_time = time.perf_counter()
svm_model = svm.SVC(probability=True, random_state=1).fit(X_train_scaled, y_train)  
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(svm_model)
save_results('SVM', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df


Train Accuracy:  0.8946813001266357
Test Accuracy:  0.8957365977205572
confusion_matrix:
 [[1910   77]
 [ 170  212]]
roc_auc_score:  0.8432751365551477
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.73      0.55      0.63       382

    accuracy                           0.90      2369
   macro avg       0.83      0.76      0.79      2369
weighted avg       0.89      0.90      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928


In [310]:
start_time = time.perf_counter()
svm_model = svm.SVC(probability=True, random_state=1, class_weight='balanced').fit(X_train_scaled, y_train)  # class_weight='balanced'  
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(svm_model)
save_results('SVM (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8655550865344027
Test Accuracy:  0.8682988602785986
confusion_matrix:
 [[1765  222]
 [  90  292]]
roc_auc_score:  0.8722468295227881
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.89      0.92      1987
           1       0.57      0.76      0.65       382

    accuracy                           0.87      2369
   macro avg       0.76      0.83      0.79      2369
weighted avg       0.89      0.87      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469


## Naive Bayes

In [ ]:
# start_time = time.perf_counter()
# gaussian_nb = GaussianNB().fit(X_train_scaled, y_train)
# end_time = time.perf_counter()
# execution_time = end_time - start_time
# accuracy, recall, precision, roc_auc = evaluate(gaussian_nb)
# save_results('Gaussian NB', accuracy, recall, precision, roc_auc, execution_time)
# results_df

Train Accuracy:  0.2514774166314901
Test Accuracy:  0.24947235120303926
confusion_matrix:
 [[ 212 1775]
 [   3  379]]
roc_auc_score:  0.8522740746791316
classification_report:
               precision    recall  f1-score   support

           0       0.99      0.11      0.19      1987
           1       0.18      0.99      0.30       382

    accuracy                           0.25      2369
   macro avg       0.58      0.55      0.25      2369
weighted avg       0.86      0.25      0.21      2369



,accuracy,recall,precision,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.882835,0.174067
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.884035,0.105618
SVM,0.896159,0.554974,0.736111,0.852601,16.339033
SVM (balanced),0.862389,0.769634,0.552632,0.872045,22.493133
Gaussian NB,0.249472,0.992147,0.175952,0.852274,0.026750


In [311]:
start_time = time.perf_counter()
bernoulli_nb = BernoulliNB().fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(bernoulli_nb)
save_results('Bernoulli NB', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.86935415787252
Test Accuracy:  0.8729421696918531
confusion_matrix:
 [[1785  202]
 [  99  283]]
roc_auc_score:  0.8794494054284788
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.90      0.92      1987
           1       0.58      0.74      0.65       382

    accuracy                           0.87      2369
   macro avg       0.77      0.82      0.79      2369
weighted avg       0.89      0.87      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419


## Bagging Models

In [312]:
start_time = time.perf_counter()
rf = RandomForestClassifier(random_state=1).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(rf)
save_results('Random Forest', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9994723512030392
Test Accuracy:  0.8910932883073026
confusion_matrix:
 [[1900   87]
 [ 171  211]]
roc_auc_score:  0.8629389460814667
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.71      0.55      0.62       382

    accuracy                           0.89      2369
   macro avg       0.81      0.75      0.78      2369
weighted avg       0.88      0.89      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
Random Forest,0.891093,0.552356,0.708054,0.620588,0.862939,1.747622


In [313]:
start_time = time.perf_counter()
rf = RandomForestClassifier(random_state=1, class_weight='balanced').fit(X_train_scaled, y_train)  # class_weight='balanced'
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(rf)
save_results('Random Forest (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9995778809624314
Test Accuracy:  0.8894048121570283
confusion_matrix:
 [[1899   88]
 [ 174  208]]
roc_auc_score:  0.8680217223470885
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.70      0.54      0.61       382

    accuracy                           0.89      2369
   macro avg       0.81      0.75      0.77      2369
weighted avg       0.88      0.89      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
Random Forest,0.891093,0.552356,0.708054,0.620588,0.862939,1.747622
Random Forest (balanced),0.889405,0.544503,0.702703,0.613569,0.868022,1.581796


## Boosting Models

### Gradient Boosting

In [314]:
start_time = time.perf_counter()
gb = GradientBoostingClassifier(random_state=1).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(gb)
save_results('Gradient Boosting', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9057619248628113
Test Accuracy:  0.897002954833263
confusion_matrix:
 [[1908   79]
 [ 165  217]]
roc_auc_score:  0.8907940882753604
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.73      0.57      0.64       382

    accuracy                           0.90      2369
   macro avg       0.83      0.76      0.79      2369
weighted avg       0.89      0.90      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
Random Forest,0.891093,0.552356,0.708054,0.620588,0.862939,1.747622
Random Forest (balanced),0.889405,0.544503,0.702703,0.613569,0.868022,1.581796
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890794,1.797643


In [315]:
weights = compute_sample_weight(class_weight='balanced', y=y_train)

start_time = time.perf_counter()
gb = GradientBoostingClassifier(random_state=1).fit(X_train_scaled, y_train, sample_weight=weights)  # Introduce class weights as GradientBoostingClassifier does not have a built-in class_weight parameter
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(gb)
save_results('Gradient Boosting (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8670325031658928
Test Accuracy:  0.8611228366399325
confusion_matrix:
 [[1742  245]
 [  84  298]]
roc_auc_score:  0.8918935120165894
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.88      0.91      1987
           1       0.55      0.78      0.64       382

    accuracy                           0.86      2369
   macro avg       0.75      0.83      0.78      2369
weighted avg       0.89      0.86      0.87      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
Random Forest,0.891093,0.552356,0.708054,0.620588,0.862939,1.747622
Random Forest (balanced),0.889405,0.544503,0.702703,0.613569,0.868022,1.581796
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890794,1.797643
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891894,2.214213


### XGBoost

In [316]:
start_time = time.perf_counter()
xgb_model = XGBClassifier(objective='binary:logistic', eval_metric='aucpr', random_state=1).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(xgb_model)
save_results('XGBoost', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9585268045588856
Test Accuracy:  0.8915154073448712
confusion_matrix:
 [[1904   83]
 [ 174  208]]
roc_auc_score:  0.8867633070455342
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.71      0.54      0.62       382

    accuracy                           0.89      2369
   macro avg       0.82      0.75      0.78      2369
weighted avg       0.88      0.89      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
Random Forest,0.891093,0.552356,0.708054,0.620588,0.862939,1.747622
Random Forest (balanced),0.889405,0.544503,0.702703,0.613569,0.868022,1.581796
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890794,1.797643
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891894,2.214213
XGBoost,0.891515,0.544503,0.714777,0.618128,0.886763,0.459835


In [317]:
scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1])

start_time = time.perf_counter()
xgb_model = XGBClassifier(objective='binary:logistic', eval_metric='aucpr', random_state=1, scale_pos_weight=scale_pos_weight).fit(X_train_scaled, y_train)  # "balanced" approach for XGBoost
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(xgb_model)
save_results('XGBoost (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9466019417475728
Test Accuracy:  0.8640776699029126
confusion_matrix:
 [[1778  209]
 [ 113  269]]
roc_auc_score:  0.8786174268873332
classification_report:
               precision    recall  f1-score   support

           0       0.94      0.89      0.92      1987
           1       0.56      0.70      0.63       382

    accuracy                           0.86      2369
   macro avg       0.75      0.80      0.77      2369
weighted avg       0.88      0.86      0.87      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
Random Forest,0.891093,0.552356,0.708054,0.620588,0.862939,1.747622
Random Forest (balanced),0.889405,0.544503,0.702703,0.613569,0.868022,1.581796
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890794,1.797643
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891894,2.214213
XGBoost,0.891515,0.544503,0.714777,0.618128,0.886763,0.459835


### LightGBM

In [318]:
start_time = time.perf_counter()
lgbm = lgb.LGBMClassifier(objective='binary', random_state=1, verbose=0).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(lgbm)
save_results('LightGBM', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9359434360489658
Test Accuracy:  0.897002954833263
confusion_matrix:
 [[1906   81]
 [ 163  219]]
roc_auc_score:  0.8909719459207361
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.73      0.57      0.64       382

    accuracy                           0.90      2369
   macro avg       0.83      0.77      0.79      2369
weighted avg       0.89      0.90      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
Random Forest,0.891093,0.552356,0.708054,0.620588,0.862939,1.747622
Random Forest (balanced),0.889405,0.544503,0.702703,0.613569,0.868022,1.581796
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890794,1.797643
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891894,2.214213
XGBoost,0.891515,0.544503,0.714777,0.618128,0.886763,0.459835


In [319]:
start_time = time.perf_counter()
lgbm = lgb.LGBMClassifier(objective='binary', random_state=1, verbose=0, class_weight='balanced').fit(X_train_scaled, y_train)  # class_weight='balanced'
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(lgbm)
save_results('LightGBM (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8933094132545378
Test Accuracy:  0.8611228366399325
confusion_matrix:
 [[1747  240]
 [  89  293]]
roc_auc_score:  0.8895826800907469
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.88      0.91      1987
           1       0.55      0.77      0.64       382

    accuracy                           0.86      2369
   macro avg       0.75      0.82      0.78      2369
weighted avg       0.89      0.86      0.87      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
Random Forest,0.891093,0.552356,0.708054,0.620588,0.862939,1.747622
Random Forest (balanced),0.889405,0.544503,0.702703,0.613569,0.868022,1.581796
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890794,1.797643
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891894,2.214213
XGBoost,0.891515,0.544503,0.714777,0.618128,0.886763,0.459835


### LightGBM (Categorical)

In [194]:
for col in ['CustomerType', 'TrafficSource', 'GeographicRegion', 'SpecialDayProximity']:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

X_train.info()

<class 'pandas.DataFrame'>
Index: 9476 entries, 10685 to 9603
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   CustomerType         9476 non-null   category
 1   SpecialDayProximity  9476 non-null   category
 2   ExitRate             9476 non-null   float64 
 3   PageValue            9476 non-null   float64 
 4   TrafficSource        9476 non-null   category
 5   GeographicRegion     9476 non-null   category
 6   BounceRate           9476 non-null   float64 
 7   ProductPageTime      9476 non-null   float64 
 8   has_page_value       9476 non-null   int64   
 9   PageValue_log        9476 non-null   float64 
 10  ProductPageTime_log  9476 non-null   float64 
dtypes: category(4), float64(6), int64(1)
memory usage: 629.6 KB


In [196]:
start_time = time.perf_counter()
lgbm = lgb.LGBMClassifier(objective='binary', random_state=1, verbose=0).fit(X_train, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc = evaluate_(lgbm)
save_results('LightGBM', accuracy, recall, precision, roc_auc, execution_time)
results_df

Train Accuracy:  0.9363655550865344
Test Accuracy:  0.8923596454200085
confusion_matrix:
 [[1905   82]
 [ 173  209]]
roc_auc_score:  0.8859813921379016
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.72      0.55      0.62       382

    accuracy                           0.89      2369
   macro avg       0.82      0.75      0.78      2369
weighted avg       0.88      0.89      0.89      2369



,accuracy,recall,precision,roc_auc,time_taken (s)
LightGBM,0.89236,0.54712,0.718213,0.885981,0.709402


In [197]:
start_time = time.perf_counter()
lgbm = lgb.LGBMClassifier(objective='binary', random_state=1, verbose=0, class_weight='balanced').fit(X_train, y_train)  # class_weight='balanced'
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc = evaluate_(lgbm)
save_results('LightGBM (balanced)', accuracy, recall, precision, roc_auc, execution_time)
results_df

Train Accuracy:  0.8996411988180667
Test Accuracy:  0.8628113127902068
confusion_matrix:
 [[1752  235]
 [  90  292]]
roc_auc_score:  0.8883686369780538
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.88      0.92      1987
           1       0.55      0.76      0.64       382

    accuracy                           0.86      2369
   macro avg       0.75      0.82      0.78      2369
weighted avg       0.89      0.86      0.87      2369



,accuracy,recall,precision,roc_auc,time_taken (s)
LightGBM,0.892360,0.547120,0.718213,0.885981,0.709402
LightGBM (balanced),0.862811,0.764398,0.554080,0.888369,0.415996


# Finetuning

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

scale_features = ['ExitRate', 'PageValue', 'BounceRate', 'ProductPageTime', 'PageValue_log', 'ProductPageTime_log']

preprocessor = ColumnTransformer(
    transformers=[('scale', StandardScaler(), scale_features)],
    remainder='passthrough'
)

# =====================
# Logistic Regression
# =====================
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=1))
])

lr_params = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__l1_ratio': [0, 0.25, 0.5, 0.75, 1],  # 0 = L2, 1 = L1
    'model__solver': ['saga'],
}

lr_search = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring='f1', n_jobs=-1)
lr_search.fit(X_train, y_train)

# =====================
# LightGBM
# =====================
lgbm_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', lgb.LGBMClassifier(objective='binary', class_weight='balanced', random_state=1, verbose=-1))
])

lgbm_params = {
    'model__n_estimators': [100, 200, 500],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth': [3, 5, 7, -1],
    'model__num_leaves': [15, 31, 63],
    'model__min_child_samples': [5, 10, 20],
    'model__subsample': [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
}

lgbm_search = RandomizedSearchCV(lgbm_pipe, lgbm_params, n_iter=50, cv=cv, scoring='f1', n_jobs=-1, random_state=1)
lgbm_search.fit(X_train, y_train)

# =====================
# Random Forest
# =====================
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=1))
])

rf_params = {
    'model__n_estimators': [100, 200, 500],
    'model__max_depth': [5, 10, 15, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2'],
}

rf_search = RandomizedSearchCV(rf_pipe, rf_params, n_iter=50, cv=cv, scoring='f1', n_jobs=-1, random_state=1)
rf_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=1))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [5, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategi

In [13]:
# =====================
# Compare results (df_cleaned, dropna)
# =====================
for name, search in [('LR', lr_search), ('LightGBM', lgbm_search), ('RF', rf_search)]:
    print(f"\n{name}:")
    print(f"  Best CV F1: {search.best_score_:.4f}")
    print(f"  Best params: {search.best_params_}")
    y_pred = search.best_estimator_.predict(X_test)
    print(f"  Test F1: {f1_score(y_test, y_pred):.4f}")
    print(f"  Test AUC: {roc_auc_score(y_test, search.best_estimator_.predict_proba(X_test)[:,1]):.4f}")


LR:
  Best CV F1: 0.6714
  Best params: {'model__C': 0.001, 'model__l1_ratio': 1, 'model__solver': 'saga'}
  Test F1: 0.6647
  Test AUC: 0.8706

LightGBM:
  Best CV F1: 0.6677
  Best params: {'model__subsample': 1.0, 'model__num_leaves': 31, 'model__n_estimators': 100, 'model__min_child_samples': 20, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}
  Test F1: 0.6721
  Test AUC: 0.9130

RF:
  Best CV F1: 0.6740
  Best params: {'model__n_estimators': 200, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 'log2', 'model__max_depth': None}


c:\Users\kaiho\Downloads\Python\aiap23-low-kai-hon-213I\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\kaiho\Downloads\Python\aiap23-low-kai-hon-213I\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Test F1: 0.6756
  Test AUC: 0.9048


In [14]:
  # =====================
  # Feature Importance
  # =====================
results = {'LR': lr_search, 'LightGBM': lgbm_search, 'RF': rf_search}

for name, search in results.items():
    best = search.best_estimator_
    model = best.named_steps['model']
    feature_names = best.named_steps['preprocessor'].get_feature_names_out()

    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_[0])

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    print(f"\n{name} — Top 10 Features:")
    for i, row in importance_df.head(10).iterrows():
        print(f"  {i+1:>2}. {row['feature']:<45} {row['importance']:.4f}")


LR — Top 10 Features:
   1. scale__PageValue_log                          0.8250
   2. scale__ExitRate                               0.0000
   3. scale__PageValue                              0.0000
   4. scale__BounceRate                             0.0000
   5. scale__ProductPageTime                        0.0000
   6. scale__ProductPageTime_log                    0.0000
   7. remainder__SpecialDayProximity                0.0000
   8. remainder__has_page_value                     0.0000
   9. remainder__CustomerType_Other                 0.0000
  10. remainder__CustomerType_Returning_Visitor     0.0000

LightGBM — Top 10 Features:
   1. scale__ExitRate                               112.0000
   2. scale__PageValue                              104.0000
   3. scale__ProductPageTime                        78.0000
   4. scale__BounceRate                             75.0000
   5. scale__ProductPageTime_log                    72.0000
   6. remainder__SpecialDayProximity                53.0

### Fillna

In [344]:
# =====================
# Compare results (df_fillna)
# =====================
for name, search in [('LR', lr_search), ('LightGBM', lgbm_search), ('RF', rf_search)]:
    print(f"\n{name}:")
    print(f"  Best CV F1: {search.best_score_:.4f}")
    print(f"  Best params: {search.best_params_}")
    y_pred = search.best_estimator_.predict(X_test)
    print(f"  Test F1: {f1_score(y_test, y_pred):.4f}")
    print(f"  Test AUC: {roc_auc_score(y_test, search.best_estimator_.predict_proba(X_test)[:,1]):.4f}")


LR:
  Best CV F1: 0.6529
  Best params: {'model__C': 0.1, 'model__l1_ratio': 0.75, 'model__solver': 'saga'}
  Test F1: 0.6591
  Test AUC: 0.8845

LightGBM:
  Best CV F1: 0.6504
  Best params: {'model__subsample': 0.7, 'model__num_leaves': 15, 'model__n_estimators': 100, 'model__min_child_samples': 20, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1.0}
  Test F1: 0.6584
  Test AUC: 0.8833

RF:
  Best CV F1: 0.6557
  Best params: {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 15}
  Test F1: 0.6589
  Test AUC: 0.8897


c:\Users\kaiho\Downloads\Python\aiap23-low-kai-hon-213I\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\kaiho\Downloads\Python\aiap23-low-kai-hon-213I\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# Results

## Cleaned (dropna)

In [116]:
# df_cleaned (dropna) - 8 main features, scaled
results_df_one_hot_encoded = results_df.copy()
results_df_one_hot_encoded.sort_values(by='recall', ascending=False)

,accuracy,recall,precision,roc_auc,time_taken
Gaussian NB,0.258715,0.993289,0.178852,0.756095,0.053961
Gradient Boosting (balanced),0.862200,0.828859,0.550111,0.907713,2.335976
LightGBM (balanced),0.858932,0.795302,0.544828,0.903219,0.364694
SVM (balanced),0.868192,0.755034,0.571066,0.880495,23.363499
Logistic Regression (balanced),0.873094,0.721477,0.589041,0.892247,0.093757
XGBoost (balanced),0.866013,0.704698,0.570652,0.885066,0.547902
LightGBM,0.890523,0.570470,0.699588,0.903041,0.345912
Bernoulli NB,0.877996,0.570470,0.639098,0.859996,0.041701
Gradient Boosting,0.892702,0.560403,0.716738,0.911509,2.240867
XGBoost,0.885076,0.550336,0.680498,0.894195,0.503074


In [225]:
# df_cleaned (dropna) - 8 main features, scaled, Sparse (no drop first)
results_df_one_hot_encoded_sparse = results_df.copy()
results_df_one_hot_encoded_sparse.sort_values(by='roc_auc', ascending=False)

,accuracy,recall,precision,roc_auc,time_taken
Gradient Boosting,0.896514,0.570470,0.732759,0.912512,NaN
Gradient Boosting (balanced),0.863290,0.825503,0.552809,0.909975,NaN
LightGBM,0.897059,0.590604,0.724280,0.906428,NaN
LightGBM (balanced),0.850218,0.778523,0.526077,0.902007,NaN
Random Forest (balanced),0.885621,0.516779,0.700000,0.893144,NaN
Random Forest,0.880719,0.523490,0.669528,0.892359,NaN
Logistic Regression (balanced),0.873094,0.721477,0.589041,0.892133,NaN
XGBoost,0.886710,0.567114,0.681452,0.890943,NaN
XGBoost (balanced),0.862745,0.718121,0.560209,0.885140,NaN
Logistic Regression,0.881808,0.375839,0.783217,0.878394,NaN


In [108]:
# LightGBM (Categorical)
results_df_lightgbm = results_df.copy()
results_df_lightgbm

,accuracy,recall,precision,roc_auc,time_taken
LightGBM,0.889978,0.563758,0.700000,0.905848,NaN
LightGBM (balanced),0.865468,0.802013,0.559719,0.901763,NaN


In [292]:
# df_cleaned (dropna) - 8 main features, scaled, Feature engineered
results_df_one_hot_encoded_feature_eng_ = results_df.copy()
results_df_one_hot_encoded_feature_eng_.sort_values(by='f1_score', ascending=False)

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression (balanced),0.873638,0.822148,0.577830,0.678670,0.899093,0.131594
Bernoulli NB,0.873094,0.805369,0.578313,0.673212,0.893661,0.036400
SVM (balanced),0.863834,0.825503,0.554054,0.663073,0.892491,26.210920
Gradient Boosting (balanced),0.862200,0.828859,0.550111,0.661312,0.908038,3.647691
LightGBM (balanced),0.864924,0.808725,0.557870,0.660274,0.903936,0.504642
Gradient Boosting,0.893246,0.563758,0.717949,0.631579,0.911570,3.224498
LightGBM,0.892157,0.563758,0.711864,0.629213,0.902747,0.479900
XGBoost (balanced),0.863290,0.697987,0.563686,0.623688,0.890512,0.725433
Random Forest,0.886710,0.570470,0.680000,0.620438,0.890999,2.123305
Random Forest (balanced),0.889434,0.546980,0.705628,0.616257,0.890329,2.083011


## Fillna

In [227]:
# Fillna - Unscaled, Feature engineered
results_df_fillna_one_hot_encoded_feature_eng_unscaled = results_df.copy()
results_df_fillna_one_hot_encoded_feature_eng_unscaled.sort_values(by='roc_auc', ascending=False)

,accuracy,recall,precision,roc_auc,time_taken (s)
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.891879,3.810738
Gradient Boosting,0.897003,0.568063,0.733108,0.890751,4.134995
LightGBM,0.891938,0.560209,0.708609,0.890627,0.567467
LightGBM (balanced),0.864500,0.769634,0.557875,0.886783,0.369360
Logistic Regression,0.886872,0.570681,0.677019,0.883618,10.027087
Logistic Regression (balanced),0.872098,0.764398,0.578218,0.883436,18.114235
XGBoost,0.893626,0.539267,0.730496,0.883315,1.202737
XGBoost (balanced),0.862811,0.717277,0.558045,0.880776,0.577001
Bernoulli NB,0.872098,0.764398,0.578218,0.874807,0.068940
Random Forest,0.890671,0.562827,0.700326,0.865660,3.492228


In [262]:
# Fillna - Scaled, Feature engineered
results_df_fillna_one_hot_encoded_feature_eng = results_df.copy()
results_df_fillna_one_hot_encoded_feature_eng.sort_values(by='f1_score', ascending=False)

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.058180
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.161197
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891861,3.639692
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,54.935445
LightGBM (balanced),0.862389,0.764398,0.553030,0.641758,0.886178,0.253794
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890773,3.778366
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,35.429912
LightGBM,0.894470,0.549738,0.729167,0.626866,0.889748,0.337930
XGBoost (balanced),0.863233,0.706806,0.560166,0.625000,0.875351,1.180660
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.768305


In [198]:
# Fillna - LightGBM (Categorical)
results_df_lightgbm = results_df.copy()
results_df_lightgbm

,accuracy,recall,precision,roc_auc,time_taken (s)
LightGBM,0.892360,0.547120,0.718213,0.885981,0.709402
LightGBM (balanced),0.862811,0.764398,0.554080,0.888369,0.415996


In [320]:
# Fillna - Scaled, Feature engineered, drop original features after log transformation
results_df_fillna_one_hot_encoded_feature_eng_drop = results_df.copy()
results_df_fillna_one_hot_encoded_feature_eng_drop.sort_values(by='f1_score', ascending=False)

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression (balanced),0.869987,0.764398,0.572549,0.654709,0.883136,0.141105
Bernoulli NB,0.872942,0.740838,0.583505,0.652826,0.879449,0.020419
SVM (balanced),0.868299,0.764398,0.568093,0.651786,0.872247,18.875469
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891894,2.214213
LightGBM,0.897003,0.573298,0.730000,0.642229,0.890972,0.267223
LightGBM (balanced),0.861123,0.767016,0.549719,0.640437,0.889583,0.260653
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890794,1.797643
SVM,0.895737,0.554974,0.733564,0.631893,0.843275,12.293928
XGBoost (balanced),0.864078,0.704188,0.562762,0.625581,0.878617,0.300060
Logistic Regression,0.888561,0.570681,0.685535,0.622857,0.883038,0.048918


# Feature Importance

In [180]:
def feature_importance(importance, X):
    feature_names = X.columns.tolist()

    # Create a DataFrame for easy viewing
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance
    })

    # Sort by importance
    importance_df = importance_df.sort_values(by='Importance', ascending=False)
    return importance_df

## Fillna - One-hot encoded, Scaled

In [182]:
feature_importance(lr.coef_[0], X_train_scaled)

,Feature,Importance
17,TrafficSource_8.0,1.210805
7,has_page_value,1.204275
20,TrafficSource_11.0,1.076292
25,TrafficSource_16.0,1.033371
4,PageValue_log,0.898600
14,TrafficSource_5.0,0.886520
29,TrafficSource_20.0,0.766140
19,TrafficSource_10.0,0.603866
11,TrafficSource_2.0,0.542439
3,ProductPageTime,0.239268


In [184]:
feature_importance(gb.feature_importances_, X_train_scaled)

,Feature,Importance
4,PageValue_log,0.410917
1,PageValue,0.409043
3,ProductPageTime,0.044452
0,ExitRate,0.031397
5,ProductPageTime_log,0.031020
6,SpecialDayProximity,0.013596
2,BounceRate,0.013575
11,TrafficSource_2.0,0.013404
17,TrafficSource_8.0,0.009085
9,CustomerType_Returning_Visitor,0.005169


In [187]:
feature_importance(lgbm.feature_importances_, X_train_scaled)

,Feature,Importance
0,ExitRate,765
3,ProductPageTime,585
2,BounceRate,433
5,ProductPageTime_log,254
1,PageValue,232
4,PageValue_log,117
11,TrafficSource_2.0,72
6,SpecialDayProximity,68
17,TrafficSource_8.0,49
31,GeographicRegion_3,48


## Fillna - LightGBM (Categorical)

In [199]:
feature_importance(lgbm.feature_importances_, X_train)

,Feature,Importance
7,ProductPageTime,937
2,ExitRate,910
6,BounceRate,442
3,PageValue,346
4,TrafficSource,143
0,CustomerType,95
5,GeographicRegion,86
1,SpecialDayProximity,39
8,has_page_value,2
9,PageValue_log,0


## Fillna - One-hot encoded, Unscaled

In [220]:
feature_importance(lr.coef_[0], X_train)

,Feature,Importance
25,TrafficSource_16.0,1.294122
17,TrafficSource_8.0,1.244507
5,has_page_value,1.210348
20,TrafficSource_11.0,1.081611
14,TrafficSource_5.0,0.942665
29,TrafficSource_20.0,0.764380
6,PageValue_log,0.718988
19,TrafficSource_10.0,0.645209
11,TrafficSource_2.0,0.580351
13,TrafficSource_4.0,0.093314


In [221]:
feature_importance(gb.feature_importances_, X_train)

,Feature,Importance
2,PageValue,0.414993
6,PageValue_log,0.405046
7,ProductPageTime_log,0.043156
4,ProductPageTime,0.032118
1,ExitRate,0.031744
0,SpecialDayProximity,0.013596
3,BounceRate,0.013590
11,TrafficSource_2.0,0.013404
17,TrafficSource_8.0,0.009085
9,CustomerType_Returning_Visitor,0.005169


In [222]:
feature_importance(lgbm.feature_importances_, X_train)

,Feature,Importance
4,ProductPageTime,804
1,ExitRate,754
3,BounceRate,446
2,PageValue,333
11,TrafficSource_2.0,84
0,SpecialDayProximity,64
17,TrafficSource_8.0,48
9,CustomerType_Returning_Visitor,48
31,GeographicRegion_3,47
30,GeographicRegion_2,36


# END